# 4.5 LoRA / LoRA

冻结原有权重，只训练两个很小的矩阵。手写 LoRA 层，并用它微调 GPT-2。

Freeze the original weights and train only two very small matrices. Write a LoRA layer by hand and fine-tune GPT-2 with it.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 手写 LoRALinear / LoRALinear by hand

给一个冻结的 `nn.Linear` 加上 LoRA。测试会检查四件事，即初始化时输出与原层相同、只有 A 和 B 可训练、它能学会一个隐藏的秩为 2 的权重改动，以及合并后得到输出相同的普通线性层。

Add LoRA to a frozen `nn.Linear`. The tests check four things, namely that the output equals that of the original layer at initialization, that only A and B are trainable, that it can learn a hidden weight change of rank 2, and that merging gives an ordinary linear layer with identical outputs.

In [ ]:
import math
import torch
import torch.nn as nn


class LoRALinear(nn.Module):
    """A frozen nn.Linear plus a trainable low-rank update.

    output = base(x) + (alpha / r) * x @ A^T @ B^T
    A: (r, in_features), small random values.   B: (out_features, r), zeros.
    """

    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        # TODO 1: freeze the parameters of base (requires_grad_(False))
        # TODO 2: create self.A as nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        #         and    self.B as nn.Parameter(torch.zeros(base.out_features, r))
        raise NotImplementedError

    def forward(self, x):
        # TODO 3
        raise NotImplementedError

    def delta_weight(self):
        """The update as a full matrix of the same shape as base.weight."""
        # TODO 4
        raise NotImplementedError

    def merge(self):
        """Return a plain nn.Linear whose weight is base.weight + delta_weight (same bias)."""
        # TODO 5
        raise NotImplementedError


def lora_param_count(d_in, d_out, r):
    """Trainable parameters of one LoRA update."""
    # TODO 6
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
import torch.nn as nn

torch.manual_seed(0)
d_in, d_out, r = 64, 48, 4
base = nn.Linear(d_in, d_out)
w0 = base.weight.detach().clone()
layer = LoRALinear(base, r=r, alpha=8)
x = torch.randn(10, d_in)

# 1. at initialisation nothing changes
assert torch.allclose(layer(x), base(x)), "B starts at zero, so the output must equal the frozen layer's"
assert layer.delta_weight().abs().max() == 0

# 2. only A and B are trainable
trainable = {n for n, p in layer.named_parameters() if p.requires_grad}
assert trainable == {"A", "B"}, f"trainable parameters: {trainable}"
n_train = sum(p.numel() for p in layer.parameters() if p.requires_grad)
assert n_train == lora_param_count(d_in, d_out, r) == r * (d_in + d_out)
assert lora_param_count(768, 2304, 8) == 24576 and 768 * 2304 == 1769472, "72 times fewer than the full matrix of GPT-2's c_attn"

# 3. the layer can learn a low-rank change of the weights
target_delta = torch.randn(d_out, 2) @ torch.randn(2, d_in) * 0.1          # a rank-2 change
y = x.new_empty(0)
X = torch.randn(512, d_in)
Y = X @ (w0 + target_delta).T + base.bias.detach()
opt = torch.optim.Adam([layer.A, layer.B], lr=1e-2)
for _ in range(800):
    loss = ((layer(X) - Y) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
assert loss.item() < 1e-4, f"final loss {loss.item():.2e}"
assert torch.equal(base.weight, w0), "the frozen weight must not move"
assert torch.linalg.matrix_rank(layer.delta_weight(), atol=1e-4) <= r, "the update has rank at most r"
assert torch.allclose(layer.delta_weight(), target_delta, atol=2e-2), "the learned update should match the hidden one"

# 4. merging gives an ordinary layer with identical outputs, so inference costs nothing extra
merged = layer.merge()
assert isinstance(merged, nn.Linear) and not hasattr(merged, "A")
assert torch.allclose(merged(x), layer(x), atol=1e-5)
print(f"m45_lora_linear: all tests passed ({n_train} trainable of {d_in * d_out + d_out} parameters)")

<details><summary><b>提示 1 / Hint 1</b></summary>

冻结用 `for p in self.base.parameters(): p.requires_grad_(False)`。

Freeze with `for p in self.base.parameters(): p.requires_grad_(False)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`nn.Linear` 的权重形状是 `(out, in)`，所以 `A` 是 `(r, in)`，`B` 是 `(out, r)`。前向中 LoRA 的部分是 `x @ self.A.T @ self.B.T`，完整的修正矩阵是 `self.B @ self.A`。

The weight of `nn.Linear` has shape `(out, in)`, so `A` is `(r, in)` and `B` is `(out, r)`. The LoRA part of the forward pass is `x @ self.A.T @ self.B.T` and the full correction matrix is `self.B @ self.A`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

合并时新建一个 `nn.Linear`，在 `torch.no_grad()` 下把 `base.weight + delta_weight()` 和偏置复制进去。

For merging create a new `nn.Linear` and, under `torch.no_grad()`, copy `base.weight + delta_weight()` and the bias into it.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch
import torch.nn as nn


class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))       # zero, so the update starts at 0

    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)

    def delta_weight(self):
        return self.scale * (self.B @ self.A)

    def merge(self):
        merged = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None)
        with torch.no_grad():
            merged.weight.copy_(self.base.weight + self.delta_weight())
            if self.base.bias is not None:
                merged.bias.copy_(self.base.bias)
        return merged


def lora_param_count(d_in, d_out, r):
    return r * (d_in + d_out)
```

</details>

## 2. 用 LoRA 微调 GPT-2 并与全量微调对比 / Fine-tune GPT-2 with LoRA and compare with full fine-tuning

给 GPT-2 每一层注意力的投影矩阵加上自己写的 LoRA，用 4.2 中同样的 8 条问答训练 12 步。测试会检查可训练参数只占 0.24%，并且与 PEFT 库的统计一致，模型学会了按格式回答，原有权重逐位未变，移除 LoRA 后模型完全恢复。运行后请把打印出的困惑度变化与 4.2 的结果对比。

Add your own LoRA to the attention projection of every GPT-2 layer and train for 12 steps on the same 8 question-answer pairs as in 4.2. The tests check that only 0.24% of the parameters are trainable, in agreement with the count of the PEFT library, that the model learns to answer in the format, that the original weights are unchanged bit for bit, and that removing the LoRA restores the model completely. Compare the printed change in perplexity with the result of 4.2.

需要联网下载模型。/ Downloads a model.

In [ ]:
# 来自练习 m42_sft 的参考实现，本题会用到 / reference code from exercise m42_sft, used below
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def format_prompt(question):
    return f"<|user|>\n{question}\n<|assistant|>\n"


def build_example(tok, question, answer):
    """One training example. Returns (input_ids, labels), two lists of equal length.

    The full text is  format_prompt(question) + f"ANSWER: {answer}." + tok.eos_token
    labels equals input_ids on the answer part (including the end token) and is -100 on the prompt part.
    """
    prompt_ids = tok(format_prompt(question)).input_ids
    answer_ids = tok(f"ANSWER: {answer}." + tok.eos_token).input_ids
    return prompt_ids + answer_ids, [-100] * len(prompt_ids) + answer_ids


def collate(examples, pad_id):
    """Pad a list of (input_ids, labels) to one batch. Nothing to change."""
    n = max(len(i) for i, _ in examples)
    ids = torch.full((len(examples), n), pad_id)
    labels = torch.full((len(examples), n), -100)
    attn = torch.zeros((len(examples), n), dtype=torch.long)
    for r, (i, l) in enumerate(examples):
        ids[r, :len(i)] = torch.tensor(i); labels[r, :len(l)] = torch.tensor(l); attn[r, :len(i)] = 1
    return ids, attn, labels


def sft_train(model, tok, pairs, steps=12, lr=1e-4):
    """Supervised fine-tuning on the whole set of pairs as one batch per step.

    Returns the list of loss values. Use torch.optim.AdamW and
    model(input_ids=..., attention_mask=..., labels=...).loss
    """
    ids, attn, labels = collate([build_example(tok, q, a) for q, a in pairs], tok.eos_token_id)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    losses = []
    for _ in range(steps):
        loss = model(input_ids=ids, attention_mask=attn, labels=labels).loss
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    model.eval()
    return losses


@torch.no_grad()
def answer(model, tok, question, max_new_tokens=8):
    """Greedy continuation of the prompt, decoded. Nothing to change."""
    ids = tok(format_prompt(question), return_tensors="pt").input_ids
    out = model.generate(ids, do_sample=False, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:])


@torch.no_grad()
def perplexity(model, tok, text):
    ids = tok(text, return_tensors="pt").input_ids
    return math.exp(model(ids, labels=ids).loss.item())

In [ ]:
import math
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

# This exercise reuses build_example, collate, answer and perplexity from module 4.2.
# GPT-2 stores its attention projection in a "Conv1D" layer:  y = x @ weight + bias,
# with weight of shape (in_features, out_features), the transpose of nn.Linear.


class LoRAConv1D(nn.Module):
    """LoRA around one of GPT-2's Conv1D layers:  y = base(x) + (alpha / r) * x @ A @ B"""

    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        d_in, d_out = base.weight.shape
        self.base, self.scale = base, alpha / r
        # TODO 1: A (d_in, r) small random (randn * 0.01),  B (r, d_out) zeros, both nn.Parameter
        raise NotImplementedError

    def forward(self, x):
        # TODO 2
        raise NotImplementedError


def add_lora(model, r=8, alpha=16):
    """Freeze every parameter of GPT-2, then wrap the c_attn layer of all 12 blocks. Returns the model."""
    # TODO 3: for p in model.parameters(): p.requires_grad_(False)
    #         for block in model.transformer.h: block.attn.c_attn = LoRAConv1D(block.attn.c_attn, r, alpha)
    raise NotImplementedError


def count_parameters(model):
    """Returns (trainable, total)."""
    # TODO 4
    raise NotImplementedError


def lora_train(model, tok, pairs, steps=12, lr=2e-3):
    """Like sft_train from module 4.2, but the optimizer only receives the trainable parameters.

    Returns the list of losses.
    """
    # TODO 5
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()
original = {n: p.detach().clone() for n, p in model.named_parameters()}
ids = tok("The capital of France is", return_tensors="pt").input_ids
with torch.no_grad():
    logits_before = model(ids).logits

model = add_lora(model, r=8, alpha=16)
with torch.no_grad():
    assert torch.allclose(model(ids).logits, logits_before, atol=1e-5), "with B = 0 the model must behave exactly as before"
trainable, total = count_parameters(model)
assert trainable == 12 * 8 * (768 + 2304) == 294912, f"trainable parameters: {trainable}"
assert trainable / total < 0.003, "less than 0.3% of the model is trained"

# The PEFT library builds the same thing: identical number of trainable parameters.
peft_model = get_peft_model(AutoModelForCausalLM.from_pretrained("openai-community/gpt2"),
                            LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"], fan_in_fan_out=True))
assert sum(p.numel() for p in peft_model.parameters() if p.requires_grad) == trainable

pairs = [("What is the capital of France?", "Paris"), ("What is the capital of Germany?", "Berlin"),
         ("What is the capital of Italy?", "Rome"), ("What is the capital of Spain?", "Madrid"),
         ("What is the capital of Japan?", "Tokyo"), ("What is the capital of China?", "Beijing"),
         ("What is the capital of Russia?", "Moscow"), ("What is the capital of Egypt?", "Cairo")]
general = "The weather was cold and the streets were quiet as people walked home from work in the evening."
ppl_before = perplexity(model, tok, general)
losses = lora_train(model, tok, pairs, steps=12, lr=2e-3)
assert losses[-1] < 0.1 * losses[0], f"loss {losses[0]:.2f} -> {losses[-1]:.3f}"
out = answer(model, tok, "What is the capital of Poland?")
assert out.startswith("ANSWER:"), f"got {out!r}"
ppl_after = perplexity(model, tok, general)

# The pretrained weights are untouched: every original tensor is bit-for-bit the same.
now = dict(model.named_parameters())
for n, p in original.items():
    key = n.replace("attn.c_attn.", "attn.c_attn.base.")
    assert torch.equal(now[key], p), f"{n} changed"
# So removing the adapters restores the original model exactly.
for block in model.transformer.h:
    block.attn.c_attn = block.attn.c_attn.base
with torch.no_grad():
    assert torch.allclose(model(ids).logits, logits_before, atol=1e-5)
assert abs(perplexity(model, tok, general) - ppl_before) < 1e-3
print("m45_lora_gpt2: all tests passed")
print(f"  trainable {trainable:,} of {total:,} parameters ({100 * trainable / total:.2f}%)")
print(f"  held-out answer: {out!r}")
print(f"  perplexity on ordinary text: {ppl_before:.1f} -> {ppl_after:.1f} with the adapters, back to {ppl_before:.1f} after removing them")

<details><summary><b>提示 1 / Hint 1</b></summary>

GPT-2 的 `Conv1D` 权重形状是 `(输入, 输出)`，与 `nn.Linear` 相反。所以这里 `A` 是 `(d_in, r)`，`B` 是 `(r, d_out)`，前向是 `x @ self.A @ self.B`，不需要转置。

The weight of GPT-2's `Conv1D` has shape `(in, out)`, the reverse of `nn.Linear`. Here `A` is therefore `(d_in, r)` and `B` is `(r, d_out)`, and the forward pass is `x @ self.A @ self.B` without transposes.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`add_lora` 中先冻结全部参数，再替换每层的 `c_attn`。顺序不能反，否则新建的 A 和 B 也会被冻结。

In `add_lora` freeze all parameters first and then replace the `c_attn` of every layer. The order matters, or the newly created A and B would be frozen as well.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

训练循环与 4.2 的 `sft_train` 相同，区别只是优化器只接收 `requires_grad` 为真的参数。

The training loop is that of `sft_train` from 4.2, except that the optimizer receives only the parameters with `requires_grad` set.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

class LoRAConv1D(nn.Module):
    """LoRA around one of GPT-2's Conv1D layers:  y = base(x) + (alpha / r) * x @ A @ B"""

    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        d_in, d_out = base.weight.shape
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(d_in, r) * 0.01)
        self.B = nn.Parameter(torch.zeros(r, d_out))

    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A @ self.B)


def add_lora(model, r=8, alpha=16):
    """Freeze every parameter of GPT-2, then wrap the c_attn layer of all 12 blocks. Returns the model."""
    for p in model.parameters():
        p.requires_grad_(False)
    for block in model.transformer.h:
        block.attn.c_attn = LoRAConv1D(block.attn.c_attn, r, alpha)
    return model


def count_parameters(model):
    """Returns (trainable, total)."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, sum(p.numel() for p in model.parameters())


def lora_train(model, tok, pairs, steps=12, lr=2e-3):
    """Like sft_train from module 4.2, but the optimizer only receives the trainable parameters.

    Returns the list of losses.
    """
    ids, attn, labels = collate([build_example(tok, q, a) for q, a in pairs], tok.eos_token_id)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    model.train()
    losses = []
    for _ in range(steps):
        loss = model(input_ids=ids, attention_mask=attn, labels=labels).loss
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    model.eval()
    return losses
```

</details>

## 3. LoRA 参数量与概念题 / Parameter counts and concept questions on LoRA

在纸上完成。
(a) 权重矩阵为 $4096 \times 4096$，取 $r = 16$。全量微调和 LoRA 的可训练参数各是多少，相差多少倍？
(b) 证明 $BA$ 的秩不超过 $r$。
(c) 如果把 $A$ 和 $B$ 都初始化为 0，会发生什么？
(d) 一个 70 亿参数的基础模型要适配 10 个任务。全量微调和 LoRA 各需要保存多少参数？假设每个 LoRA 占基础模型的 0.2%。

Work on paper.
(a) The weight matrix is $4096 \times 4096$ and $r = 16$. How many trainable parameters do full fine-tuning and LoRA have, and by what factor do they differ?
(b) Show that the rank of $BA$ does not exceed $r$.
(c) What happens if both $A$ and $B$ are initialized to 0?
(d) A base model of 7 billion parameters is to be adapted to 10 tasks. How many parameters must be stored with full fine-tuning and with LoRA? Assume each LoRA takes 0.2% of the base model.

<details><summary><b>提示 1 / Hint 1</b></summary>

(b) $BA$ 的每一列都是 $B$ 的各列的线性组合。

(b) Every column of $BA$ is a linear combination of the columns of $B$.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 写出两个矩阵的梯度。

(c) Write the gradients of the two matrices.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 全量微调为 $4096^2 = 16{,}777{,}216$。LoRA 为 $16 \times (4096 + 4096) = 131{,}072$。相差 128 倍。

(b) $BA$ 的每一列是 $B$ 的 $r$ 个列向量的线性组合，所以 $BA$ 的列空间包含在 $B$ 的列空间内，维数不超过 $r$。

(c) $\partial \mathcal{L} / \partial B \propto G A^{\top} = 0$，$\partial \mathcal{L} / \partial A \propto B^{\top} G = 0$。两个梯度都是 0，参数停在原地，无法学习。所以需要其中一个不为 0。

(d) 全量微调需要 10 份完整的权重，共 700 亿个参数。LoRA 需要一份基础模型加 10 个适配器，共 $70 + 10 \times 0.14 = 71.4$ 亿个参数。

---

(a) Full fine-tuning has $4096^2 = 16{,}777{,}216$. LoRA has $16 \times (4096 + 4096) = 131{,}072$. The factor is 128.

(b) Every column of $BA$ is a linear combination of the $r$ column vectors of $B$, so the column space of $BA$ lies within that of $B$ and has dimension at most $r$.

(c) $\partial \mathcal{L} / \partial B \propto G A^{\top} = 0$ and $\partial \mathcal{L} / \partial A \propto B^{\top} G = 0$. Both gradients are 0, the parameters stay where they are and nothing is learned. One of the two therefore has to be non-zero.

(d) Full fine-tuning needs 10 complete copies of the weights, 70 billion parameters in total. LoRA needs one base model plus 10 adapters, $7 + 10 \times 0.014 = 7.14$ billion parameters in total.

</details>